# Block Stacking Problem

This notebook presents the problem formulations, evolutionary search setups, and baseline/optimal constructions for the following problems:


## 29. Block Stacking Problem

### Detailed Problem Description
Let $n \geq 1$.  Let $C(n)$ be the largest displacement that the $n^{\mathrm{th}}$ block in a stack of identical rigid rectangular blocks of width $1$ can be displaced horizontally over the edge of a table, with the stack remaining stable.  More mathematically, $C(n)$ is the supremum of $x_n$ where $0 = x_0 \leq x_1 \leq \dots\leq x_n$ are real numbers subject to the constraints
$$ \frac{x_{i+1}+\dots+x_n}{n-i} < x_i + \frac{1}{2}$$
for all $0 \leq i < n$.  What is $C(n)$?


## AlphaEvolve Search Configuration

**Prompt**

Blocks stacking problem

Act as a research mathematician and optimization specialist.

GOAL:
Your task is to write a python function get_positions(n) that for any given integer n outputs a list of horizontal positions of n rectangular blocks of width 1 stacked on top of each other, maximizing the horizontal overhang of the top block over the edge of the table while satisfying stability constraints.

Specifically, the Python function you have to provide has the following
signature:

def get_positions(n: int) -> list[float] | np.ndarray

EVALUATION:

Your construction will be scored by a function called
get_score.
The interface of get_score is:

def get_score(construction) -> float

Your list of elements will be evaluated by get_score which evaluates the stability constraints and returns the overhang of the top block. The scoring will evaluate your function on a mixture of small and large inputs n and take the average.
You may code up any search method you want, and you are allowed to call the
get_score() function as many times as you want. You have access to it,
you don't need to code up the get_score() function.
You want the score it gives you to be as large as possible!

Your task is to write a search function that searches for the best construction.
Your function will have 1000 seconds to run, and after that it has to have
returned the best construction it found. If after 1000 seconds it has not
returned anything, it will be terminated with negative infinity points. You can
use your time best if you have an outer loop of the form
"while time.time() - start_time < 1000:" or similar, just don't forget to define
the "start_time" variable early in your program.


### Initial Program (Baseline/Search Seed)

In [ ]:
def get_positions(n: int) -> list[float]:
    # Simple linear stack (suboptimal)
    return [i * 0.1 for i in range(n)]

### Evolved Code by AlphaEvolve

In [ ]:
FLOAT_TOLERANCE = 1e-9

def get_positions_recurrence(n: int) -> list[float]:
    # First evolved code: using suffix sums recursive relation
    if n == 0:
        return []
    x_values = [0.0] * n
    current_sum_x_suffix = 0.0
    for i in range(n - 2, -1, -1):
        count_suffix = n - 1 - i
        x_values[i] = current_sum_x_suffix / count_suffix - 0.5
        current_sum_x_suffix += x_values[i]
    total_x_sum = current_sum_x_suffix
    P_upper_bound_target = 0.5 - total_x_sum / n
    P = P_upper_bound_target - 2 * FLOAT_TOLERANCE
    P_lower_bound = 0.0
    for x_val in x_values:
        P_lower_bound = max(P_lower_bound, -x_val)
    if P <= P_lower_bound + FLOAT_TOLERANCE:
        P = max(P, P_lower_bound + 2 * FLOAT_TOLERANCE)
    positions = [(P + x_val) for x_val in x_values]
    return positions

def get_positions_harmonic(n: int) -> list[float]:
    # Second evolved code: simplified using harmonic numbers
    if n == 0:
        return []
    harmonic_numbers = [0.0] * (n + 1)
    for j in range(1, n + 1):
        harmonic_numbers[j] = harmonic_numbers[j-1] + 1.0 / j
    q_values = [0.0] * n
    for k in range(n):
        q_values[k] = 0.5 * (harmonic_numbers[n] - harmonic_numbers[n - k - 1]) - 0.5 - 2 * FLOAT_TOLERANCE
    positions = [q + 0.5 for q in q_values]
    return positions

### Evaluator Function

In [ ]:
def get_positions_score(positions: list[float]) -> float:
    n = len(positions)
    if n == 0: return 0.0
    if n == 1:
        if positions[0] - 0.5 >= 0.0 - FLOAT_TOLERANCE: return -1.0
        return positions[0]
    sum_all = 0.0
    for k in range(n):
        sum_all += (positions[k] - 0.5)
    sum_all_avg = sum_all / n
    if sum_all_avg >= 0.0 - FLOAT_TOLERANCE: return -1.0
    upper_sum = 0.0
    upper_count = 0.0
    if n > 1:
        upper_sum = positions[n-1] - 0.5
        upper_count = 1
        for i in range(n - 2, -1, -1):
            upper_sum_avg = upper_sum / upper_count
            lb = positions[i] - 1.0
            ub = positions[i]
            if not (lb - FLOAT_TOLERANCE <= upper_sum_avg <= ub + FLOAT_TOLERANCE): return -1.0
            upper_sum += (positions[i] - 0.5)
            upper_count += 1.0
    return positions[-1]

### Data Verification and Results

In [ ]:
n = 5
pos_rec = get_positions_recurrence(n)
pos_harm = get_positions_harmonic(n)
print("Score for recurrence positions:", get_positions_score(pos_rec))
print("Score for harmonic positions:  ", get_positions_score(pos_harm))
print("Theoretical optimum (0.5 * H_5):", 0.5 * sum(1.0/i for i in range(1, 6)))